## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore")


def _open_with_retries(url_or_urls, *, filters=None, storage_options=None, max_attempts=3, base_sleep=1):
    """Open a grib2io xarray dataset with retry on transient open failures."""
    kw = dict(
        engine="grib2io",
        use_icechunk=True,
        storage_options=storage_options or {},
        filters=filters,
        max_workers=2,
        network_timeout=300,
        max_concurrent_requests=8,
        combine="nested",
        concat_dim="valid_time",
        chunks={},
    )
    for attempt in range(1, max_attempts + 1):
        try:
            if isinstance(url_or_urls, (list, tuple)):
                return xr.open_mfdataset(url_or_urls, **kw)
            return xr.open_dataset(url_or_urls, **kw)
        except Exception as exc:
            if attempt == max_attempts:
                raise
            sleep_s = base_sleep * attempt
            print(f"Transient open failure ({type(exc).__name__}) on attempt {attempt}/{max_attempts}; retrying in {sleep_s}s...")
            time.sleep(sleep_s)


def compute(arr, *, max_attempts=5, base_sleep=2):
    """Trigger Dask computation with automatic retry on transient network errors."""
    for attempt in range(1, max_attempts + 1):
        try:
            return arr.compute()
        except Exception as exc:
            msg = str(exc).lower()
            transient = (
                "timeout" in msg
                or "timed out" in msg
                or "connect" in msg
                or "dns error" in msg
                or "nodename nor servname" in msg
                or "icechunkerror" in type(exc).__name__.lower()
            )
            if (not transient) or attempt == max_attempts:
                raise
            sleep_s = base_sleep**attempt
            print(f"Transient error on attempt {attempt}/{max_attempts}, retrying in {sleep_s}s: {exc}")
            time.sleep(sleep_s)


# Optional notebook-only dependency for future VirtualiZarr/object-store workflows.
try:
    from obstore.store import S3Store
    from obspec_utils.registry import ObjectStoreRegistry

    OBSTORE_AVAILABLE = True
    print("Optional obstore dependencies are available.")
except ImportError:
    OBSTORE_AVAILABLE = False

In [ ]:
BUCKET = "noaa-gefs-pds"
YEAR = "2025"  # any YYYY present in the bucket
GRID = "0p25"  # 0.25-degree global grid
CYCLE = "00"  # 00Z analysis
FORECAST = "f000"  # analysis hour (zero-lead)

# More tolerant network settings for remote NOAA S3 reads via s3fs/botocore.
storage_options = {
    "anon": True,
    "config_kwargs": {
        "connect_timeout": 30,
        "read_timeout": 120,
        "retries": {"max_attempts": 10, "mode": "adaptive"},
    },
}

# Filter to 2-metre temperature only — skips all other variables during scanning.
# typeOfFirstFixedSurface=103 → "height above ground (m)"; level=2 → 2 m
T2M_FILTERS = {"shortName": "totAOD550", "typeOfFirstFixedSurface": 10}

# Build URLs for the selected year.
year_start = pd.to_datetime(f"{YEAR}-01-01")
year_end = pd.to_datetime(f"{YEAR}-12-31")
dates = pd.date_range(year_start, year_end, freq="D")
PERIOD_LABEL = YEAR
urls = [f"s3://{BUCKET}/gefs.{d.strftime('%Y%m%d')}/{CYCLE}/chem/pgrb2ap25/gefs.chem.t{CYCLE}z.a2d_0p25.{FORECAST}.grib2" for d in dates]
# urls2 = [f"s3://{BUCKET}/gfs.{d.strftime('%Y%m%d')}/{CYCLE}/atmos/gfs.t{CYCLE}z.pgrb2.{GRID}.f120" for d in dates]
print(f"Period: {PERIOD_LABEL}  ({len(urls)} files)")
print("First:", urls[0])
print("Last: ", urls[-1])

In [ ]:
import time

t0 = time.perf_counter()
ds_sample = xr.open_dataset(
    urls[0],
    engine="grib2io",
    use_icechunk=True,
    storage_options=storage_options,
    filters=T2M_FILTERS,
    max_workers=2,
    network_timeout=300,
    max_concurrent_requests=2,
    chunks={},
)
print(f"Opened in {time.perf_counter() - t0:.1f}s")
print(ds_sample)

In [ ]:
t0 = time.perf_counter()
T2M_VAR = "totAOD550"
ds = _open_with_retries(urls, filters=T2M_FILTERS, storage_options=storage_options)
elapsed = time.perf_counter() - t0
print(f"Prepared {ds.sizes.get('valid_time', 0)} valid_time values in {elapsed:.1f}s (Dask graph, not computed)")
print(f"Expected valid_time count: {len(dates)}")

with ProgressBar():
    ds_final = compute(ds)
elapsed = time.perf_counter() - t0
print(f"Loaded {ds_final.sizes.get('valid_time', 0)} valid_time values in {elapsed:.1f}s (fully computed)")

In [ ]:
print("Computing annual mean AOD550 ...")

aod_mean = compute(ds_final[T2M_VAR].mean("valid_time"))

lats = aod_mean.coords.get("latitude", aod_mean.coords.get("y", None))
lons = aod_mean.coords.get("longitude", aod_mean.coords.get("x", None))

fig, ax = plt.subplots(figsize=(14, 6))
if lats is not None and lons is not None:
    pcm = ax.pcolormesh(lons, lats, aod_mean.squeeze(), cmap="rainbow", shading="auto")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
else:
    pcm = ax.imshow(aod_mean.squeeze(), origin="upper", cmap="rainbow", aspect="auto")
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")

plt.colorbar(pcm, ax=ax, label="AOD at 550 nm", shrink=0.7)
ax.set_title(f"GEFS 00Z — Mean Total AOD at 550 nm\n{PERIOD_LABEL}  (00Z, {GRID})")
plt.tight_layout()
plt.show()

In [ ]:
import hvplot.xarray  # noqa: F401 - registers .hvplot accessor on xarray objects
import geoviews as gv
import holoviews as hv
import panel as pn

gv.extension("bokeh")
pn.extension()

print(ds_final)
# Use the lazy Dask-backed dataset so each step loads one day on demand.
aod_lazy = ds_final[T2M_VAR].squeeze()
print(aod_lazy)

# Normalize dimension names for plotting.
rename = {}
if "y" in aod_lazy.dims and "latitude" not in aod_lazy.dims:
    rename["y"] = "latitude"
if "x" in aod_lazy.dims and "longitude" not in aod_lazy.dims:
    rename["x"] = "longitude"
if rename:
    aod_lazy = aod_lazy.rename(rename).persist()

# Coordinate cleanup:
aod_plot = aod_lazy.assign_coords(
    latitude=aod_lazy["latitude"].clip(min=-90.0, max=90.0),
    longitude=((aod_lazy["longitude"] + 180.0) % 360.0) - 180.0,
)

# Color scale based on already computed annual mean; keeps stepping lazy.
vmax = float(aod_mean.squeeze().quantile(0.99)) * 3

# Ensure predictable valid_time order before building controls.
aod_plot = aod_plot.sortby("valid_time")
times = pd.to_datetime(aod_plot["valid_time"].values)

step_slider = pn.widgets.IntSlider(
    name="Time step",
    start=0,
    end=len(times) - 1,
    value=0,
    step=1,
    width=420,
)
prev_btn = pn.widgets.Button(name="◀ Previous", button_type="light", width=120)
next_btn = pn.widgets.Button(name="Next ▶", button_type="light", width=120)
time_label = pn.pane.Markdown(sizing_mode="stretch_width")


def _update_time_label(event=None):
    t = times[step_slider.value]
    time_label.object = f"**valid_time:** {t:%Y-%m-%d %H:%M}"


def _go_prev(event):
    if step_slider.value > step_slider.start:
        step_slider.value -= 1


def _go_next(event):
    if step_slider.value < step_slider.end:
        step_slider.value += 1


prev_btn.on_click(_go_prev)
next_btn.on_click(_go_next)
step_slider.param.watch(_update_time_label, "value")
_update_time_label()


@pn.depends(step_slider.param.value)
def _frame(idx):
    frame = aod_plot.isel(valid_time=idx)
    return frame.hvplot.quadmesh(
        x="longitude",
        y="latitude",
        cmap="YlGnBu",
        clim=(0, 1),
        clabel="AOD at 550 nm",
        title=f"GEFS 00Z - Total AOD at 550 nm  {PERIOD_LABEL}  ({GRID})",
        width=900,
        height=450,
        geo=True,
        project=False,
        coastline=True,
        rasterize=True,
    )


pn.Column(
    pn.Row(prev_btn, next_btn, step_slider),
    time_label,
    _frame,
    sizing_mode="stretch_width",
)

In [ ]:
print(ds_final[T2M_VAR])